# Group XX - Training Notebook
# Krones Bottle-Base Inspection — Multi-task EfficientNetV2-S + LightGBM Stacker

**What this notebook does (read me first):**

This is our **training** notebook. It trains the deep-learning model and the stacker, and saves
everything the *evaluation* notebook needs to make predictions later. We deliberately keep training
and evaluation in **separate** notebooks because the competition runs only the evaluation notebook
in a fresh environment — so this one produces the artifacts, and the evaluation one consumes them.

**The pipeline, in plain words:**
1. Look at the data (EDA) — class balance, image sizes, defect categories, example images.
2. Crop each bottle to its base using the ROI box from the annotations.
3. Train an EfficientNetV2-S that predicts **two things at once**: (a) is the bottle FAULTY,
   and (b) which of the 26 defect categories are present. Predicting the categories too (a
   "multi-task" auxiliary head) forces the network to learn *why* a bottle is faulty, which makes
   it much better at the main yes/no question.
4. Collect leakage-free out-of-fold (OOF) predictions across CV folds.
5. Train a small **LightGBM stacker** that combines the network's outputs into a final, sharper
   prediction, and search for the best decision threshold.
6. Save the model weights, the stacker, the threshold, and the config so evaluation can reload them.

**Artifacts saved to `/kaggle/working/` (the evaluation notebook reloads these):**
- `model_fold{0,1,2}.pt` — the trained network weights, one per CV fold
- `stacker.txt` — the trained LightGBM stacker
- `best_threshold.json` — the decision threshold + which prediction mode to use
- `oof.csv`, `oof_aux.npy` — out-of-fold predictions (for analysis / re-tuning the stacker)


## 0 — Imports and reproducibility
We fix every random seed so the run is reproducible.

In [ ]:
import os, sys, json, glob, gc, time, random, statistics
from pathlib import Path
import numpy as np
import pandas as pd
import cv2                       # fast image read + ROI crop
import matplotlib.pyplot as plt  # EDA plots
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, recall_score, precision_score,
                             confusion_matrix, roc_auc_score)
from tqdm.auto import tqdm

# timm gives us pretrained EfficientNet; lightgbm is the stacker. Install if missing.
try:
    import timm
except ImportError:
    os.system(f"{sys.executable} -m pip install -q timm"); import timm
try:
    import lightgbm as lgb
except ImportError:
    os.system(f"{sys.executable} -m pip install -q lightgbm"); import lightgbm as lgb

def seed_everything(seed=42):
    """Fix all RNGs (python, numpy, torch) so results are reproducible."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True   # lets cuDNN pick fast kernels for our fixed image size

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1 — Configuration

Every knob lives in one dictionary so it's easy to see and change. The comments explain *why*
each value is what it is.

In [ ]:
CFG = {
    "seed": 42,
    # Competition data location on Kaggle. (The evaluation notebook will swap in the hidden test.)
    "data_dir": "/kaggle/input/competitions/1st-krones-vision-ai-challenge",
    "out_dir":  "/kaggle/working",          # where we save model + artifacts

    # ── Backbone / image ──
    # EfficientNetV2-S pretrained on ImageNet-21k then 1k: strong features, still fast/small.
    "backbone": "tf_efficientnetv2_s.in21k_ft_in1k",
    "img_size": 448,        # bottle bases are detailed; 448 keeps small defects visible
    "in_chans": 1,          # the photos are grayscale, so 1 channel (not 3) — cleaner + faster
    "mean": 0.449, "std": 0.226,   # normalization stats for a single grayscale channel
    "roi_margin": 0.06,     # 6% padding around the ROI box so we don't clip the base edge

    # ── Multi-task ──
    # Weight of the auxiliary (26-category) loss relative to the main FAULTY/GOOD loss.
    # 0.4 means "care about the main task most, but learn the categories as a strong helper".
    "aux_weight": 0.4,

    # ── Cross-validation / training ──
    "n_folds": 3,
    "train_folds": [0, 1, 2],   # train ALL folds — the stacker needs OOF over the whole train set
    "epochs": 5,
    "batch_size": 16,
    "lr": 1.5e-4,
    "weight_decay": 1e-4,
    "num_workers": 2,
    "pin_memory": True,
    "use_amp": True,            # mixed precision: faster + less memory on GPU

    # ── Optional upgrades (off = simplest model). Flip to experiment. ──
    "use_advanced_arch": False, # True adds GeM pooling + a squeeze-excite gate
    "gem_p": 3.0, "head_hidden": 512, "head_dropout": 0.3,
    "use_tta": False,           # True averages 4 flips/rotations at predict time (slower, small gain)

    # ── Stacker (LightGBM) ──
    "lgb_params": dict(objective="binary", n_estimators=500, learning_rate=0.03,
                       num_leaves=31, min_child_samples=60, subsample=0.8, subsample_freq=1,
                       colsample_bytree=0.8, reg_lambda=1.0, verbosity=-1, random_state=42),
    "threshold_grid": (0.05, 0.95, 0.0025),   # search FAULTY threshold on this fine grid
}
seed_everything(CFG["seed"])

DATA_DIR = Path(CFG["data_dir"]); OUT = Path(CFG["out_dir"]); OUT.mkdir(exist_ok=True)
TRAIN_IMG_DIR = DATA_DIR/"train_images"
TEST_IMG_DIR  = DATA_DIR/"test_images"
TRAIN_CSV     = DATA_DIR/"train.csv"
ANN_JSON      = DATA_DIR/"train_annotations.json"
BOTTLETYPE_CSV= DATA_DIR/"bottletypes.csv"
print("Data dir exists:", DATA_DIR.exists())

## 2 — Load the labels and metadata

`train.csv` has the binary label (`target`: 1 = FAULTY, 0 = GOOD) per image.
`bottletypes.csv` (optional) tells us the bottle model — useful as a feature and for fair CV splits.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
print("train.csv shape:", train_df.shape)
print(train_df.head())

# Attach bottle type if the file exists (helps CV stratification + is a stacker feature)
if BOTTLETYPE_CSV.exists():
    bt = pd.read_csv(BOTTLETYPE_CSV)
    train_bt = bt[bt["split"] == "train"][["image_id", "bottle_type"]]
    train_df = train_df.merge(train_bt, on="image_id", how="left")
    test_type_map = dict(zip(bt["image_id"], bt["bottle_type"]))   # for test-time feature
else:
    train_df["bottle_type"] = "unknown"; test_type_map = {}
train_df["bottle_type"] = train_df["bottle_type"].fillna("unknown")
BOTTLE_TYPES = sorted(train_df["bottle_type"].unique())
print("Bottle types:", BOTTLE_TYPES)

## 3 — Exploratory Data Analysis (EDA)

Before modelling we look at the data: how balanced are the classes, how big are the images, what
defect categories exist and how often, and what do the bottles actually look like. These checks
catch surprises early (e.g. a class that barely appears) and motivate our design choices.

In [ ]:
# ── 3.1 Class balance ──
counts = train_df["target"].value_counts().sort_index()
print("Label counts:  GOOD(0) =", int(counts.get(0,0)), "| FAULTY(1) =", int(counts.get(1,0)))
print("FAULTY fraction: {:.1%}".format(train_df["target"].mean()))

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar(["GOOD","FAULTY"], [counts.get(0,0), counts.get(1,0)], color=["#4c72b0","#c44e52"])
ax[0].set_title("Class balance"); ax[0].set_ylabel("number of images")
# bottle-type distribution
bt_counts = train_df["bottle_type"].value_counts()
ax[1].bar(range(len(bt_counts)), bt_counts.values, color="#55a868")
ax[1].set_xticks(range(len(bt_counts))); ax[1].set_xticklabels(bt_counts.index, rotation=45, ha="right")
ax[1].set_title("Images per bottle type")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 Defect categories from the COCO annotations ──
# train_annotations.json holds per-defect boxes. Category id 22 is the ROI (the base region),
# everything else is a defect type. We count how often each defect appears.
with open(ANN_JSON) as f:
    train_ann = json.load(f)

ROI_CAT = 22
cat_names = {c["id"]: c["name"] for c in train_ann["categories"]}
from collections import Counter
defect_counter = Counter()
for a in train_ann["annotations"]:
    if a["category_id"] != ROI_CAT:
        defect_counter[cat_names[a["category_id"]]] += 1

defect_series = pd.Series(defect_counter).sort_values(ascending=True)
print("Number of defect categories:", len(defect_series))
plt.figure(figsize=(9, 8))
plt.barh(defect_series.index, defect_series.values, color="#8172b3")
plt.title("Defect category frequency (very imbalanced!)")
plt.xlabel("number of annotated instances"); plt.tight_layout(); plt.show()
print("\nRarest 5 defects:\n", defect_series.head(5))
print("\nMost common 5 defects:\n", defect_series.tail(5))

In [ ]:
# ── 3.3 Image size check (a few images) ──
sample_files = train_df["image_id"].sample(min(50, len(train_df)), random_state=0).tolist()
shapes = []
for f in sample_files:
    im = cv2.imread(str(TRAIN_IMG_DIR/f), cv2.IMREAD_GRAYSCALE)
    if im is not None: shapes.append(im.shape)
shapes = np.array(shapes)
print("Sampled image heights:", np.unique(shapes[:,0]))
print("Sampled image widths :", np.unique(shapes[:,1]))
print("-> images share a consistent size; good, our ROI crop + resize will be uniform.")

In [ ]:
# ── 3.4 Look at actual bottles: GOOD vs FAULTY ──
def show_examples(df, label, n=4):
    sub = df[df["target"]==label]["image_id"].sample(n, random_state=1).tolist()
    fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
    for i, f in enumerate(sub):
        im = cv2.imread(str(TRAIN_IMG_DIR/f), cv2.IMREAD_GRAYSCALE)
        ax[i].imshow(im, cmap="gray"); ax[i].axis("off")
        ax[i].set_title(("GOOD" if label==0 else "FAULTY")+f"\n{f[:12]}", fontsize=8)
    plt.tight_layout(); plt.show()

print("Examples of GOOD bottles:");   show_examples(train_df, 0)
print("Examples of FAULTY bottles:"); show_examples(train_df, 1)
print("Observation: defects are often small/faint vs the whole base -> motivates ROI crop + the")
print("aux head (per-category supervision) so the model learns subtle defect appearances.")

## 4 — ROI boxes and auxiliary (26-category) targets

We crop each image to its bottle base using the ROI box (COCO category 22). We also build a
26-dim multi-label vector per image (which defect categories are present) — these are the
**auxiliary targets** the network learns alongside the main GOOD/FAULTY label.

In [ ]:
# Map COCO image-id -> filename, then collect ROI boxes (category 22)
tr_id2file = {im["id"]: Path(im["file_name"]).name for im in train_ann["images"]}
tr_roi = {tr_id2file[a["image_id"]]: a["bbox"]
          for a in train_ann["annotations"] if a["category_id"] == ROI_CAT}
print("Train images with an ROI box:", len(tr_roi))

# Build the 26-category auxiliary target vectors (exclude ROI category 22)
cat_ids = sorted(c["id"] for c in train_ann["categories"] if c["id"] != ROI_CAT)
N_AUX = len(cat_ids)
cat_to_idx = {c: i for i, c in enumerate(cat_ids)}
print("Number of auxiliary defect categories (N_AUX):", N_AUX)

aux_targets = {}   # filename -> 26-dim 0/1 vector
for a in train_ann["annotations"]:
    if a["category_id"] == ROI_CAT:
        continue
    fname = tr_id2file.get(a["image_id"])
    if fname is None:
        continue
    vec = aux_targets.setdefault(fname, np.zeros(N_AUX, dtype=np.float32))
    vec[cat_to_idx[a["category_id"]]] = 1.0

del train_ann; gc.collect()
print("Images with at least one defect annotation:", len(aux_targets))

## 5 — ROI crop function and the PyTorch Dataset

`crop_roi` makes a **square** crop centred on the ROI box (so the resize doesn't distort the
circular base) and reflects the border if the square pokes outside the image. The `BottleDataset`
loads a grayscale image, crops it, optionally augments it, and returns it as a normalized tensor
along with the labels.

In [ ]:
def crop_roi(img, rx, ry, rw, rh, margin, out_size):
    """Square crop around the ROI box (+margin), reflect-pad if needed, resize to out_size."""
    h, w = img.shape[:2]
    cx, cy = rx + rw/2.0, ry + rh/2.0           # ROI centre
    s = max(rw, rh) * (1.0 + 2.0*margin)         # square side = longest ROI edge + margin
    x0, y0 = int(round(cx - s/2)), int(round(cy - s/2))
    x1, y1 = int(round(cx + s/2)), int(round(cy + s/2))
    # how much the square sticks out of the image on each side
    px0, py0 = max(0,-x0), max(0,-y0)
    px1, py1 = max(0, x1-w), max(0, y1-h)
    crop = img[max(0,y0):min(h,y1), max(0,x0):min(w,x1)]
    if px0 or py0 or px1 or py1:                  # reflect padding looks natural at the edges
        crop = cv2.copyMakeBorder(crop, py0, py1, px0, px1, cv2.BORDER_REFLECT_101)
    # INTER_AREA when shrinking (sharper), INTER_LINEAR when enlarging
    interp = cv2.INTER_AREA if crop.shape[0] > out_size else cv2.INTER_LINEAR
    return cv2.resize(crop, (out_size, out_size), interpolation=interp)

# If an image has no ROI box, fall back to the median ROI so it still gets a sensible crop.
_rois = np.array(list(tr_roi.values()), dtype=np.float32)
FALLBACK_ROI = tuple(np.median(_rois, axis=0)) if len(_rois) else (160,147,987,722)
print("Fallback ROI (median):", [round(v,1) for v in FALLBACK_ROI])

class BottleDataset(Dataset):
    """Returns (image, target, aux_vector) for training, or (image, image_id) for test."""
    def __init__(self, ids, img_dir, roi_map, cfg, targets=None, aux=None, train_aug=False):
        self.ids = list(ids); self.img_dir = Path(img_dir); self.roi = roi_map; self.cfg = cfg
        self.targets = targets; self.aux = aux; self.train_aug = train_aug
        hit = sum(1 for f in self.ids if f in roi_map)
        print(f"  Dataset built: {hit}/{len(self.ids)} images have a per-image ROI box")
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        f = self.ids[i]
        img = cv2.imread(str(self.img_dir/f), cv2.IMREAD_GRAYSCALE)
        if img is None: raise FileNotFoundError(self.img_dir/f)
        rx, ry, rw, rh = self.roi.get(f, FALLBACK_ROI)
        img = crop_roi(img, rx, ry, rw, rh, self.cfg["roi_margin"], self.cfg["img_size"])
        # Light, label-preserving augmentation (bottle bases are rotationally symmetric)
        if self.train_aug:
            if random.random() < 0.5: img = np.fliplr(img)
            if random.random() < 0.5: img = np.flipud(img)
            k = random.randint(0,3)
            if k: img = np.rot90(img, k)
            if random.random() < 0.4:   # tiny brightness/contrast jitter
                img = np.clip(img.astype(np.float32)*random.uniform(0.92,1.08)
                              + random.uniform(-8,8), 0, 255).astype(np.uint8)
        # to tensor, scale to [0,1], normalize, add channel dim -> (1, H, W)
        x = torch.from_numpy(np.ascontiguousarray(img)).float()
        x = x.div_(255.0).sub_(self.cfg["mean"]).div_(self.cfg["std"]).unsqueeze(0)
        if self.targets is None:
            return x, f
        y = torch.tensor(float(self.targets[i]), dtype=torch.float32)
        a = torch.from_numpy(self.aux[i]) if self.aux is not None else torch.zeros(N_AUX)
        return x, y, a

## 6 — The model: `KronesNet` (one backbone, two heads)

A shared EfficientNetV2-S backbone produces a feature vector, then **two heads** read from it:
- `head` → 1 number: the FAULTY/GOOD logit (the task we are scored on)
- `aux_head` → 26 numbers: which defect categories are present (the helper task)

Training both together makes the features encode *defect-specific* information, which is the main
reason this model beats a plain binary classifier. The `use_advanced_arch` toggle optionally swaps
global pooling for **GeM** (keeps small-defect signal) and adds a squeeze-excite gate.

In [ ]:
class GeMPooling(nn.Module):
    """Generalized-mean pooling: emphasises peak activations so small defects survive pooling.
    Done in fp32 with p clamped to [1,5] to stay numerically safe under mixed precision."""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1)*p); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(1.0, 5.0)
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            xf = x.float().clamp(min=self.eps)
            out = F.avg_pool2d(xf.pow(p), kernel_size=(xf.shape[-2], xf.shape[-1])).pow(1.0/p)
        return out.view(out.shape[0], -1)

class SEGate(nn.Module):
    """Squeeze-excite: learns a per-channel importance weight (channel attention)."""
    def __init__(self, feat, reduction=16):
        super().__init__(); mid = max(feat//reduction, 16)
        self.g = nn.Sequential(nn.Linear(feat, mid), nn.ReLU(inplace=True),
                               nn.Linear(mid, feat), nn.Sigmoid())
    def forward(self, x): return x * self.g(x)

class KronesNet(nn.Module):
    """EfficientNetV2-S backbone + main head (1 logit) + aux head (26 logits)."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        self.advanced = cfg["use_advanced_arch"]
        if self.advanced:
            # keep the spatial feature map (global_pool="") so we can apply GeM ourselves
            self.encoder = timm.create_model(cfg["backbone"], pretrained=pretrained,
                                             in_chans=cfg["in_chans"], num_classes=0, global_pool="")
            feat = self.encoder.num_features
            self.gem = GeMPooling(cfg["gem_p"]); self.gate = SEGate(feat)
        else:
            # let timm do standard global average pooling -> a feature vector
            self.encoder = timm.create_model(cfg["backbone"], pretrained=pretrained,
                                             in_chans=cfg["in_chans"], num_classes=0)
            feat = self.encoder.num_features
        self.head     = nn.Linear(feat, 1)       # main: FAULTY vs GOOD
        self.aux_head = nn.Linear(feat, N_AUX)   # auxiliary: 26 defect categories
    def forward(self, x):
        if self.advanced:
            f = self.gate(self.gem(self.encoder.forward_features(x)))
        else:
            f = self.encoder(x)
        return self.head(f).squeeze(-1), self.aux_head(f)

# Build once to confirm shapes are right before training
_m = KronesNet(CFG, pretrained=False).to(DEVICE)
with torch.no_grad():
    _l, _a = _m(torch.randn(2, CFG["in_chans"], CFG["img_size"], CFG["img_size"]).to(DEVICE))
print("Build OK | main head:", tuple(_l.shape), "| aux head:", tuple(_a.shape),
      "| params:", f"{sum(p.numel() for p in _m.parameters()):,}")
del _m, _l, _a; torch.cuda.empty_cache()

## 7 — Loss, threshold search, and prediction helper

The total loss is `main_loss + aux_weight * aux_loss`. The main loss uses `pos_weight` to handle
class imbalance. `best_f1_threshold` searches a fine grid for the threshold that maximises F1 (it
can also enforce a minimum recall). `predict_feats` runs a model and returns both the main
probability and the 26 aux probabilities (these feed the stacker).

In [ ]:
# pos_weight tells BCE to pay more attention to the minority class
pos = float(train_df["target"].sum()); neg = float(len(train_df)-pos)
POS_WEIGHT = torch.tensor([neg/max(pos,1.0)], dtype=torch.float32).to(DEVICE)
main_bce = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
print("pos_weight (neg/pos):", round(float(POS_WEIGHT.item()),3))

def criterion(main_logit, aux_logit, y, aux_y):
    """Combined multi-task loss: main FAULTY/GOOD + weighted auxiliary 26-category."""
    main = main_bce(main_logit, y)
    aux  = F.binary_cross_entropy_with_logits(aux_logit, aux_y)
    return main + CFG["aux_weight"] * aux

lo, hi, step = CFG["threshold_grid"]
THRESHOLDS = np.round(np.arange(lo, hi+1e-9, step), 4)

def best_f1_threshold(y_true, probs):
    """Return (threshold, F1) maximising F1 over the grid."""
    y_true = np.asarray(y_true).astype(int); probs = np.asarray(probs)
    best_thr, best = 0.5, -1.0
    for thr in THRESHOLDS:
        s = f1_score(y_true, (probs>=thr).astype(int), zero_division=0)
        if s > best: best, best_thr = s, float(thr)
    return best_thr, best

def _views(x, on):
    """Test-time augmentation views: original + 3 flips (only if TTA is on)."""
    return [x, torch.flip(x,[3]), torch.flip(x,[2]), torch.flip(x,[2,3])] if on else [x]

@torch.no_grad()
def predict_feats(net, ids, img_dir, roi_map, cfg=CFG):
    """Run the network and return (main_probs, aux_probs). Averages TTA views if enabled."""
    dl = DataLoader(BottleDataset(ids, img_dir, roi_map, cfg), batch_size=cfg["batch_size"]*2,
                    shuffle=False, num_workers=cfg["num_workers"], pin_memory=cfg["pin_memory"])
    net = net.to(DEVICE).eval(); mains, auxs = [], []
    for x, _ in dl:
        x = x.to(DEVICE)
        mv, av = [], []
        for v in _views(x, cfg["use_tta"]):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=cfg["use_amp"] and DEVICE.type=="cuda"):
                ml, al = net(v)
            mv.append(torch.sigmoid(ml.float())); av.append(torch.sigmoid(al.float()))
        mains.append(torch.stack(mv).mean(0).cpu().numpy())
        auxs.append(torch.stack(av).mean(0).cpu().numpy())
    return np.concatenate(mains), np.concatenate(auxs)

## 8 — Cross-validation folds

We split into 3 folds, **stratified by both the label and the bottle type**, so each fold has a
similar mix. Training on 2 folds and validating on the 3rd (rotating) gives us *out-of-fold*
predictions for every training image — predictions from a model that never saw that image, which
is exactly what the stacker needs.

In [ ]:
train_df = train_df.reset_index(drop=True)
# combine label + bottle type for stratification; merge rare combos back to label-only
train_df["strat"] = train_df["target"].astype(str) + "_" + train_df["bottle_type"].astype(str)
rare = train_df["strat"].value_counts()
rare = rare[rare < CFG["n_folds"]].index
train_df.loc[train_df["strat"].isin(rare), "strat"] = train_df["target"].astype(str)

skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["strat"])):
    train_df.loc[val_idx, "fold"] = fold
print(train_df.groupby(["fold","target"]).size())

# aux target matrix aligned to train_df row order (zeros for images with no defect annotation)
AUX_MAT = np.stack([aux_targets.get(f, np.zeros(N_AUX, dtype=np.float32))
                    for f in train_df["image_id"]]).astype(np.float32)
print("Aux target matrix:", AUX_MAT.shape)

## 9 — Train each fold and collect out-of-fold (OOF) predictions

For each fold: train the network (multi-task loss), keep the epoch with the best validation F1,
then run that best model on the held-out fold to record OOF main + aux probabilities. We save one
`.pt` file per fold — these are the model weights the evaluation notebook will load.

In [ ]:
oof_main = np.full(len(train_df), np.nan, dtype=np.float64)   # main FAULTY prob per image (OOF)
oof_aux  = np.zeros((len(train_df), N_AUX), dtype=np.float32)  # 26 aux probs per image (OOF)
fold_info, model_paths = [], []
start_all = time.time()

for fold in CFG["train_folds"]:
    print("\n"+"="*70+f"\nFOLD {fold}\n"+"="*70)
    trn_m = (train_df["fold"] != fold).values; val_m = ~trn_m
    trn_idx = np.where(trn_m)[0]; val_idx = np.where(val_m)[0]

    trn_loader = DataLoader(
        BottleDataset(train_df.loc[trn_m,"image_id"], TRAIN_IMG_DIR, tr_roi, CFG,
                      targets=train_df.loc[trn_m,"target"].values, aux=AUX_MAT[trn_idx], train_aug=True),
        batch_size=CFG["batch_size"], shuffle=True,
        num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"], drop_last=True)
    val_loader = DataLoader(
        BottleDataset(train_df.loc[val_m,"image_id"], TRAIN_IMG_DIR, tr_roi, CFG,
                      targets=train_df.loc[val_m,"target"].values, aux=AUX_MAT[val_idx]),
        batch_size=CFG["batch_size"]*2, shuffle=False,
        num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"])

    net = KronesNet(CFG, pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
    scaler = torch.amp.GradScaler(enabled=CFG["use_amp"] and DEVICE.type=="cuda")

    best_f1v, best_path = -1.0, str(OUT/f"model_fold{fold}.pt")
    for epoch in range(1, CFG["epochs"]+1):
        net.train(); running = 0.0; t0 = time.time()
        for x, y, a in tqdm(trn_loader, desc=f"Fold{fold} epoch{epoch}", leave=False):
            x, y, a = x.to(DEVICE), y.to(DEVICE), a.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["use_amp"] and DEVICE.type=="cuda"):
                ml, al = net(x)
                loss = criterion(ml, al, y, a)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            running += loss.item()*x.size(0)
        scheduler.step()

        # validation F1 (main head only) to pick the best epoch
        net.eval(); P, Y = [], []
        with torch.no_grad():
            for x, y, a in val_loader:
                with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["use_amp"] and DEVICE.type=="cuda"):
                    ml, _ = net(x.to(DEVICE))
                P += torch.sigmoid(ml.float()).cpu().numpy().tolist(); Y += y.numpy().tolist()
        thr, f1v = best_f1_threshold(np.array(Y), np.array(P))
        print(f"Fold {fold} | epoch {epoch}/{CFG['epochs']} | "
              f"loss={running/len(trn_loader.dataset):.5f} | val_F1={f1v:.5f} (thr {thr:.3f}) | "
              f"{(time.time()-t0)/60:.1f} min")
        if f1v > best_f1v:
            best_f1v = f1v; torch.save(net.state_dict(), best_path); print("   saved best ->", best_path)

    # reload best, record OOF main + aux for this fold
    net.load_state_dict(torch.load(best_path, map_location=DEVICE))
    ids = train_df.loc[val_m, "image_id"].tolist()
    m_p, a_p = predict_feats(net, ids, TRAIN_IMG_DIR, tr_roi)
    oof_main[val_idx] = m_p; oof_aux[val_idx] = a_p
    fold_info.append({"fold": int(fold), "best_val_f1": float(best_f1v), "path": best_path})
    model_paths.append(best_path)
    del net, optimizer, scheduler, scaler, trn_loader, val_loader; gc.collect(); torch.cuda.empty_cache()

print("\nTotal training time (min):", round((time.time()-start_all)/60, 1))
# Save OOF for the stacker + later analysis
np.save(OUT/"oof_aux.npy", oof_aux)
train_df["prob"] = oof_main
train_df[["image_id","target","bottle_type","fold","prob"]].to_csv(OUT/"oof.csv", index=False)
print("Saved oof.csv and oof_aux.npy")

## 10 — Train the LightGBM stacker

The stacker is a small gradient-boosted tree model. Its inputs per image are:
`[ logit(main prob), the 26 aux probs, bottle-type one-hot ]`. It learns to *combine* these into a
sharper final probability — effectively a learned version of "is the FAULTY prob borderline but a
serious defect category is active?". We train it **fold-safe** (each fold predicted by trees fit on
the other folds), compare raw vs stacked vs a 50/50 blend on OOF F1, and keep whichever wins.

In [ ]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps); return np.log(p/(1-p))

# Build the stacker feature matrix
type_oh = pd.get_dummies(train_df["bottle_type"]).reindex(columns=BOTTLE_TYPES).values.astype(np.float32)
X = np.column_stack([logit(oof_main), oof_aux, type_oh]).astype(np.float32)
y = train_df["target"].values
print("Stacker feature matrix X:", X.shape, "(1 main + %d aux + %d type)" % (N_AUX, len(BOTTLE_TYPES)))

# Fold-safe OOF predictions from the stacker (no leakage)
stacked = np.zeros(len(train_df))
for fold in CFG["train_folds"]:
    tr_m = (train_df["fold"] != fold).values
    clf = lgb.LGBMClassifier(**CFG["lgb_params"]); clf.fit(X[tr_m], y[tr_m])
    stacked[~tr_m] = clf.predict_proba(X[~tr_m])[:, 1]

# Compare the three options on OOF F1
raw_thr, raw_f1 = best_f1_threshold(y, oof_main)
stk_thr, stk_f1 = best_f1_threshold(y, stacked)
blend = 0.5*oof_main + 0.5*stacked
bl_thr, bl_f1 = best_f1_threshold(y, blend)
print(f"raw    : F1 {raw_f1:.5f} (thr {raw_thr:.4f})")
print(f"stack  : F1 {stk_f1:.5f} (thr {stk_thr:.4f})")
print(f"blend  : F1 {bl_f1:.5f} (thr {bl_thr:.4f})")

mode = max([("raw",raw_f1),("stack",stk_f1),("blend",bl_f1)], key=lambda t: t[1])[0]
chosen_thr   = {"raw":raw_thr,"stack":stk_thr,"blend":bl_thr}[mode]
chosen_probs = {"raw":oof_main,"stack":stacked,"blend":blend}[mode]
preds = (chosen_probs >= chosen_thr).astype(int)
print(f"\nChosen mode: {mode} | threshold {chosen_thr:.4f}")
print(f"OOF F1 {f1_score(y,preds):.5f} | recall {recall_score(y,preds):.4f} | precision {precision_score(y,preds):.4f}")
print("Confusion matrix [rows=true GOOD/FAULTY]:\n", confusion_matrix(y, preds))

# Retrain the stacker on ALL OOF rows for test-time use, and save it + the decision config
final_clf = lgb.LGBMClassifier(**CFG["lgb_params"]); final_clf.fit(X, y)
final_clf.booster_.save_model(str(OUT/"stacker.txt"))
payload = {
    "mode": mode, "best_threshold": chosen_thr,
    "oof_f1": {"raw": raw_f1, "stack": stk_f1, "blend": bl_f1},
    "types": BOTTLE_TYPES, "n_aux": N_AUX, "cat_ids": cat_ids,
    "backbone": CFG["backbone"], "img_size": CFG["img_size"], "in_chans": CFG["in_chans"],
    "mean": CFG["mean"], "std": CFG["std"], "roi_margin": CFG["roi_margin"],
    "use_advanced_arch": CFG["use_advanced_arch"], "use_tta": CFG["use_tta"],
    "model_paths": [Path(p).name for p in model_paths],
}
json.dump(payload, open(OUT/"best_threshold.json","w"), indent=2)
print("\nSaved stacker.txt and best_threshold.json")
print(json.dumps(payload, indent=2))

## 11 — Result analysis (plots for the report)

Confusion matrix and the FAULTY-probability distribution. These show the model separates the two
classes cleanly and motivate where the remaining errors are.

In [ ]:
import itertools
cm = confusion_matrix(y, preds)
fig, ax = plt.subplots(1, 2, figsize=(12,4.5))
im = ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0,1]); ax[0].set_xticklabels(["GOOD","FAULTY"])
ax[0].set_yticks([0,1]); ax[0].set_yticklabels(["GOOD","FAULTY"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True"); ax[0].set_title(f"Confusion matrix (OOF, {mode})")
for i,j in itertools.product(range(2),range(2)):
    ax[0].text(j,i,cm[i,j],ha="center",va="center",
               color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=13)
ax[1].hist(chosen_probs[y==0], bins=40, alpha=0.6, label="GOOD", color="#4c72b0")
ax[1].hist(chosen_probs[y==1], bins=40, alpha=0.6, label="FAULTY", color="#c44e52")
ax[1].axvline(chosen_thr, color="k", ls="--", label=f"threshold {chosen_thr:.3f}")
ax[1].set_xlabel("predicted P(FAULTY)"); ax[1].set_ylabel("count")
ax[1].set_title("Score separation"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Final OOF F1:", round(f1_score(y, preds), 5),
      "| ROC-AUC:", round(roc_auc_score(y, chosen_probs), 5))

## 12 — Done

Artifacts now in `/kaggle/working/`: `model_fold0.pt`, `model_fold1.pt`, `model_fold2.pt`,
`stacker.txt`, `best_threshold.json`, `oof.csv`, `oof_aux.npy`.

**Next:** attach these as a Kaggle *Dataset*, share it (privately) with both organizers, and point
the **evaluation notebook** at them. The evaluation notebook reloads these and runs inference only —
it does not retrain.